# 10 - Feature engineering

**Objectif** : créer de nouvelles variables à partir des données existantes pour améliorer le boosting retenu au notebook 09 (511 / 573 €/m² en validation croisée aléatoire / par commune).

**Entrée** : `dvf_77_2025_enrichi.parquet`, même découpage qu'aux notebooks 08 et 09.
**Sortie** : aucun fichier.

**Règle de départ** : un modèle à arbres coupe sur des seuils, donc une transformation monotone d'une variable (log, racine) ne change rien. Seules des **combinaisons** de variables ou une **information nouvelle** peuvent aider.

**Plan**
1. Direction depuis Paris et rapports entre caractéristiques (sans fuite de données)
2. Prix médian des ventes voisines (construit avec la cible)

## 1. Direction et rapports

- `angle_paris` : direction depuis Paris (0° à l'est, 90° au nord, -90° au sud), la distance seule étant un cercle.
- `surface_par_piece` : surface habitable divisée par le nombre de pièces.
- `terrain_par_m2_bati` : terrain divisé par la surface habitable.

Aucune de ces variables n'utilise le prix : pas de risque de fuite.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GroupKFold
from sklearn.ensemble import HistGradientBoostingRegressor

df = pd.read_parquet("../data/processed/dvf_77_2025_enrichi.parquet")
variables = ["type_local", "surface_reelle_bati", "nombre_pieces_principales",
             "nb_dependances", "surface_terrain_total", "distance_paris_km",
             "latitude", "longitude"]
df_modele = df.dropna(subset=["distance_paris_km"])
X = df_modele[variables].copy()
X["type_local"] = (X["type_local"] == "Maison").astype(int)
y = df_modele["prix_m2"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
groupes = df_modele.loc[X_train.index, "code_commune"]

def scores(modele, X=X_train):
    aleatoire = -cross_val_score(modele, X, y_train, cv=5,
                                 scoring="neg_mean_absolute_error").mean()
    par_commune = -cross_val_score(modele, X, y_train, cv=GroupKFold(n_splits=5),
                                   groups=groupes, scoring="neg_mean_absolute_error").mean()
    return round(aleatoire), round(par_commune)

hgb = HistGradientBoostingRegressor(loss="absolute_error", learning_rate=0.03, max_iter=600,
                                    max_leaf_nodes=31, early_stopping=False, random_state=42)

# Nouvelles variables
lat_paris, lon_paris = 48.8534, 2.3488

def ajouter_variables(X):
    X = X.copy()
    dx = (X["longitude"] - lon_paris) * np.cos(np.radians(lat_paris))
    dy = X["latitude"] - lat_paris
    X["angle_paris"] = np.degrees(np.arctan2(dy, dx))
    X["surface_par_piece"] = X["surface_reelle_bati"] / X["nombre_pieces_principales"]
    X["terrain_par_m2_bati"] = X["surface_terrain_total"] / X["surface_reelle_bati"]
    return X

X_train_fe = ajouter_variables(X_train)
print(X_train_fe[["angle_paris", "surface_par_piece", "terrain_par_m2_bati"]].describe().round(1))

print("Référence  :", scores(hgb))
print("+ angle    :", scores(hgb, X_train_fe.drop(columns=["surface_par_piece", "terrain_par_m2_bati"])))
print("+ rapports :", scores(hgb, X_train_fe.drop(columns=["angle_paris"])))
print("+ tout     :", scores(hgb, X_train_fe))

       angle_paris  surface_par_piece  terrain_par_m2_bati
count      10964.0            10964.0              10964.0
mean         -19.2               22.5                  4.0
std           33.0                6.0                 15.6
min          -85.0                9.5                  0.0
25%          -55.2               19.0                  0.0
50%          -15.2               21.3                  1.7
75%            7.7               24.6                  5.0
max           52.5              165.0               1321.3
Référence  : (511, 573)
+ angle    : (508, 574)
+ rapports : (510, 571)
+ tout     : (509, 572)


**Lecture** : l'angle va de -85° (sud : Nemours, Fontainebleau) à 52° (nord-est : Meaux), cohérent avec un département entièrement à l'est de Paris (pas de discontinuité à ±180°). Les rapports ont quelques extrêmes (165 m² par pièce, terrain 1 321 fois le bâti) sans gêne pour un modèle à arbres. Aucun ajout ne dépasse le bruit (écarts de 1 à 3 €/m²) : le modèle reconstruisait déjà ces informations à partir de la latitude, de la longitude et des variables brutes. **Non retenu.**

## 2. Prix médian des voisins

Pour chaque vente, médiane du prix au m² des k ventes connues les plus proches (distance de haversine). Pour une vente future, son prix est inconnu : on utilise celui des ventes passées autour d'elle, comme une estimation d'agent immobilier.

Le transformateur `PrixVoisins`, placé dans un `Pipeline` :
- `fit` mémorise positions et prix des ventes d'entraînement ;
- `transform` (validation, test, ventes futures) cherche les voisines parmi l'entraînement ;
- `fit_transform` (entraînement) exclut chaque vente de ses propres voisines.

L'évaluation reste honnête : en validation, la variable est toujours calculée à partir de la partie d'entraînement seule.

In [2]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline

class PrixVoisins(BaseEstimator, TransformerMixin):
    def __init__(self, k=20):
        self.k = k

    def fit(self, X, y):
        # On mémorise les positions et les prix des ventes d'entraînement
        self.prix_ = np.asarray(y)
        coords = np.radians(X[["latitude", "longitude"]].to_numpy())
        self.voisins_ = NearestNeighbors(metric="haversine").fit(coords)
        return self

    def transform(self, X):
        # Ventes nouvelles (validation, test, futur) : voisines parmi l'entraînement
        coords = np.radians(X[["latitude", "longitude"]].to_numpy())
        idx = self.voisins_.kneighbors(coords, n_neighbors=self.k, return_distance=False)
        X = X.copy()
        X["prix_voisins"] = np.median(self.prix_[idx], axis=1)
        return X

    def fit_transform(self, X, y):
        # Ventes d'entraînement : on exclut chaque vente de ses propres voisines
        self.fit(X, y)
        coords = np.radians(X[["latitude", "longitude"]].to_numpy())
        idx = self.voisins_.kneighbors(coords, n_neighbors=self.k + 1, return_distance=False)
        idx_sans_soi = np.array([ligne[ligne != i][: self.k] for i, ligne in enumerate(idx)])
        X = X.copy()
        X["prix_voisins"] = np.median(self.prix_[idx_sans_soi], axis=1)
        return X

for k in [10, 20, 50]:
    modele = make_pipeline(PrixVoisins(k=k), hgb)
    print(f"k = {k} :", scores(modele))

k = 10 : (504, 585)
k = 20 : (506, 583)
k = 50 : (507, 581)


**Lecture** : en aléatoire, gain de 4 à 7 €/m² (504 à 507 contre 511), à peine au-dessus du bruit. Par commune, l'erreur **augmente** de 8 à 12 €/m² (581 à 585 contre 573). Explication probable : à l'entraînement, les voisines d'une vente sont surtout dans sa commune, voire dans le même immeuble ou programme (prix très proches), donc la variable est très fiable et le modèle s'y fie. Pour une commune jamais vue, les voisines viennent des communes d'à côté, et la variable est moins fiable qu'à l'entraînement. Augmenter k atténue cet écart (585 vers 581), ce qui va dans le sens de cette explication. Exclure la vente elle-même ne suffit donc pas : il faudrait aussi exclure ses quasi-doublons (même immeuble). **Non retenu.**

## Conclusion

| Variables ajoutées | Aléatoire | Par commune |
| --- | --- | --- |
| Référence (notebook 09) | 511 | 573 |
| + angle | 508 | 574 |
| + rapports | 510 | 571 |
| + angle et rapports | 509 | 572 |
| + prix des voisins (k = 10 / 20 / 50) | 504 / 506 / 507 | 585 / 583 / 581 |

- **Aucune nouvelle variable n'est retenue** : direction et rapports n'apportent rien, le prix des voisins aide à peine pour les zones connues et dégrade pour les communes nouvelles.
- **Le plateau est confirmé** : avec latitude, longitude et 600 arbres, le modèle capte déjà la localisation et les caractéristiques du bien. Le gain ne viendra que de données vraiment nouvelles (âge et état du bien, transports, plusieurs années).
- **Leçon** : une variable construite avec la cible doit avoir la même fiabilité à l'entraînement et à l'usage. Exclure la vente elle-même ne suffit pas quand des ventes quasi identiques (même immeuble) existent.